# D19 S1 A2 - 1x1 Convolution and Skip-Connection Gradients

NumPy implementation for the six required functions and the supplied verification checks.


In [1]:
import numpy as np

rng = np.random.RandomState(1)
x = rng.randn(3, 4, 4)
W = rng.randn(3, 3) * 0.2


In [2]:
def relu(x):
    return np.maximum(0, x)


def relu_grad(z):
    return (z > 0).astype(float)


def conv1x1(x, W):
    return np.einsum('oc,chw->ohw', W, x)


def forward_plain(x, W):
    return relu(conv1x1(x, W))


def forward_residual(x, W):
    return relu(conv1x1(x, W) + x)


def block_input_grad(x, W, skip=False):
    pre = conv1x1(x, W)
    if skip:
        pre = pre + x

    mask = relu_grad(pre)
    grad = np.einsum('oc,ohw->chw', W, mask)

    if skip:
        grad = grad + mask

    return grad


## Verification


In [3]:
rng = np.random.RandomState(1)
x = rng.randn(3, 4, 4)
print(conv1x1(x, np.eye(3)).shape)
print(np.allclose(conv1x1(x, np.eye(3)), x))

W0 = np.zeros((3, 3))
print(np.allclose(block_input_grad(x, W0, skip=False), 0))
print(np.allclose(block_input_grad(x, W0, skip=True), (x > 0)))


(3, 4, 4)
True
True
True


## Autograder-style checks


In [4]:
rng = np.random.RandomState(7)

# 1: output shape
x_test = rng.randn(3, 4, 5)
W_test = rng.randn(6, 3)
y_test = conv1x1(x_test, W_test)
assert y_test.shape == (6, 4, 5)

# 2: per-pixel channel mixing
co, h, w = 2, 1, 3
expected = sum(W_test[co, ci] * x_test[ci, h, w] for ci in range(3))
assert np.allclose(y_test[co, h, w], expected)

# 3: identity mixer
x_id = rng.randn(3, 4, 4)
assert np.allclose(conv1x1(x_id, np.eye(3)), x_id)

# 4: ReLU gradient mask
z = np.array([-2.0, 0.0, 1.5])
assert np.array_equal(relu_grad(z), np.array([0.0, 0.0, 1.0]))

# 5: residual forward
W_res = rng.randn(3, 3) * 0.2
assert np.allclose(forward_residual(x_id, W_res), relu(conv1x1(x_id, W_res) + x_id))

# 6: zero branch kills plain gradient
W0 = np.zeros((3, 3))
assert np.allclose(block_input_grad(x_id, W0, skip=False), 0)

# 7: skip preserves gradient when branch vanishes
assert np.allclose(block_input_grad(x_id, W0, skip=True), (x_id > 0))

# 8: near-vanishing branch has stronger residual gradient
W_small = rng.randn(3, 3) * 1e-4
g_plain = block_input_grad(x_id, W_small, skip=False)
g_res = block_input_grad(x_id, W_small, skip=True)
assert np.linalg.norm(g_res) > np.linalg.norm(g_plain)

print('All 8 checks passed.')


All 8 checks passed.
